In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub

# Download latest version
path = kagglehub.dataset_download("simuletic/cctv-shoplifting-detection-dataset-yolo-and-vlm")

print("Path to dataset files:", path)

print('Data source import complete.')


100%|██████████| 482M/482M [00:07<00:00, 66.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/simuletic/cctv-shoplifting-detection-dataset-yolo-and-vlm/versions/1
Data source import complete.


Check the data

In [ ]:
import os

for root, dirs, files in os.walk(path):
    level = root.replace(path, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")


1/
    CCTV_Shoplifting_Dataset/
        annotated_images/
        VLM-labels/
        labels/
        videos/
        images/
        annotated_videos/


In [ ]:
pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 6.9 MB/s eta 0:00:00


In [ ]:
import cv2
import os
import numpy as np


**2. Define input/output paths**

Input = your Kaggle dataset

Output = where frames will be saved

In [ ]:
root = "/kaggle/input/shoplifting-video-dataset"
output_root = "/kaggle/working/frames_dataset"
os.makedirs(output_root, exist_ok=True)


**3. Analyze Video for footfall**

In [ ]:
from pathlib import Path

DATASET_DIR = Path(
    "/root/.cache/kagglehub/datasets/simuletic/"
    "cctv-shoplifting-detection-dataset-yolo-and-vlm/versions/1"
)

test_vid = DATASET_DIR / "CCTV_Shoplifting_Dataset" / "videos" / "shoplifting3.mp4"

print(test_vid)
print(test_vid.exists())

/root/.cache/kagglehub/datasets/simuletic/cctv-shoplifting-detection-dataset-yolo-and-vlm/versions/1/CCTV_Shoplifting_Dataset/videos/shoplifting3.mp4
True


### YOLO detector + pose: estimate if a person faces the ad screen

Adjust `SCREEN_ROI` to the screen’s actual pixel coordinates before running. Two 2D models are used: a detector with ByteTrack for person IDs and a pose model for visible head landmarks. Facing-screen labels are **heuristic**, not eye-gaze or proof of attention. Side-facing heads with visible landmarks work best; frontal views, occlusion and overhead cameras frequently yield UNKNOWN. The example screen coordinates are placeholders.


In [ ]:
import cv2
import numpy as np
from collections import defaultdict
from ultralytics import YOLO

# Keep detection + ByteTrack for stable person IDs; use pose for head direction.
YOLO_detector = YOLO("yolo26n.pt")
YOLO_pose = YOLO("yolo26n-pose.pt")

MIN_DWELL_SECONDS = 2.0
MIN_LOOK_SECONDS = 0.5  # cumulative high-confidence facing-screen frames
POSE_CONFIDENCE = 0.35
MATCH_IOU_THRESHOLD = 0.30
KEYPOINT_CONFIDENCE = 0.45
# Nose must shift horizontally by this fraction of eye/ear baseline
HEAD_SHIFT_RATIO = 0.12

# Ground-plane ROI: a person's bottom-center must be inside.
ROI = np.array([[200,150],[800,150],[800,650],[200,650]], dtype=np.int32)

# IMPORTANT: EDIT THIS to the *actual advertisement screen location* in the video.
# Polygon traces the visible screen in IMAGE pixels, NOT the area where people stand.
SCREEN_ROI = np.array([[50,190],[175,190],[175,370],[50,370]], dtype=np.int32)
SCREEN_CENTER = SCREEN_ROI.astype(float).mean(axis=0)

OUTPUT_VIDEO = "/kaggle/working/footfall_dwell_pose_output.mp4"


def point_inside_roi(point, roi):
    return cv2.pointPolygonTest(roi, (float(point[0]), float(point[1])), False) >= 0


def box_iou(box_a, box_b):
    x1, y1 = np.maximum(box_a[:2], box_b[:2])
    x2, y2 = np.minimum(box_a[2:], box_b[2:])
    intersection = max(0, x2-x1) * max(0, y2-y1)
    area_a = max(0,box_a[2]-box_a[0]) * max(0,box_a[3]-box_a[1])
    area_b = max(0,box_b[2]-box_b[0]) * max(0,box_b[3]-box_b[1])
    union = area_a + area_b - intersection
    return intersection/union if union > 0 else 0


def match_pose_to_tracks(det_boxes, pose_boxes):
    """Greedy one-to-one matching by IoU. Returns detection index -> pose index."""
    candidates = sorted(
        [(box_iou(d,p),di,pi) for di,d in enumerate(det_boxes)
                               for pi,p in enumerate(pose_boxes)], reverse=True
    )
    matched = {}; used_poses = set()
    for iou, di, pi in candidates:
        if iou < MATCH_IOU_THRESHOLD: break
        if di not in matched and pi not in used_poses:
            matched[di] = pi; used_poses.add(pi)
    return matched


def facing_screen_from_pose(xy, conf, screen_center):
    """Return (True/False/None, head_point) from 2D head pose.

    COCO: nose=0, eyes=1/2, ears=3/4. Need visible symmetric pair.
    Nose offset from pair midpoint approximates LEFT/RIGHT head orientation.
    'None' is unknown, NOT looking away. No vertical-depth / true gaze inference.
    """
    def visible(k):
        return conf[k] >= KEYPOINT_CONFIDENCE and np.isfinite(xy[k]).all() and (xy[k] > 0).all()
    if not visible(0): return None, None
    nose = xy[0].astype(float)
    if visible(3) and visible(4): pair = (3,4)
    elif visible(1) and visible(2): pair = (1,2)
    else: return None, nose
    mid = (xy[pair[0]]+xy[pair[1]])/2
    baseline = float(np.linalg.norm(xy[pair[0]]-xy[pair[1]]))
    if baseline < 6: return None, nose
    head_dx = float(nose[0]-mid[0])
    screen_dx = float(screen_center[0]-nose[0])
    if abs(screen_dx) < 0.4*baseline or abs(head_dx) < HEAD_SHIFT_RATIO*baseline:
        return None, nose
    return bool(head_dx * screen_dx > 0), nose


In [ ]:
# ===================================================
# VIDEO SETUP
# ===================================================
cap = cv2.VideoCapture(str(test_vid))
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {test_vid}")
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0: raise ValueError("Invalid video FPS")
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width,height))
if not writer.isOpened(): raise RuntimeError(f"Cannot write to {OUTPUT_VIDEO}")

footfall_ids = set()
frames_inside = defaultdict(int)
previously_inside = defaultdict(bool)
look_frames = defaultdict(int)
look_ids = set()
frame_number = 0

while cap.isOpened():
    success, frame = cap.read()
    if not success: break
    frame_number += 1

    tracked = YOLO_detector.track(
        frame, persist=True, tracker="bytetrack.yaml",
        classes=[0], conf=0.30, verbose=False
    )[0]
    posed = YOLO_pose.predict(frame, conf=POSE_CONFIDENCE, verbose=False)[0]

    cv2.polylines(frame, [ROI], True, (0,255,255), 2)
    cv2.putText(frame, "Exposure ROI", tuple(ROI[0]),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0,255,255), 2)
    cv2.polylines(frame, [SCREEN_ROI], True, (255,0,255), 2)
    cv2.putText(frame, "AD SCREEN", tuple(SCREEN_ROI[0]),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,0,255), 2)

    if tracked.boxes is not None and tracked.boxes.id is not None:
        boxes = tracked.boxes.xyxy.cpu().numpy()
        ids = tracked.boxes.id.int().cpu().tolist()
        if posed.boxes is not None and posed.keypoints is not None:
            pose_boxes = posed.boxes.xyxy.cpu().numpy()
            pose_xy = posed.keypoints.xy.cpu().numpy()
            pose_conf = (posed.keypoints.conf.cpu().numpy()
                         if posed.keypoints.conf is not None
                         else np.ones(pose_xy.shape[:2]))
        else:
            pose_boxes = np.empty((0,4)); pose_xy = np.empty((0,17,2))
            pose_conf = np.empty((0,17))
        matches = match_pose_to_tracks(boxes, pose_boxes)

        for i, (box, track_id) in enumerate(zip(boxes, ids)):
            x1,y1,x2,y2 = map(int,box)
            foot_point = (int((x1+x2)/2),y2)
            inside = point_inside_roi(foot_point,ROI)
            if inside:
                frames_inside[track_id] += 1
                if not previously_inside[track_id]: footfall_ids.add(track_id)
            previously_inside[track_id] = inside
            dwell_seconds = frames_inside[track_id] / fps

            looking, head = None, None
            if i in matches:
                pi = matches[i]
                looking, head = facing_screen_from_pose(pose_xy[pi], pose_conf[pi], SCREEN_CENTER)
                # Show reliable head keypoints from YOLO pose.
                for k in [0,1,2,3,4]:
                    if pose_conf[pi,k] >= KEYPOINT_CONFIDENCE and (pose_xy[pi,k]>0).all():
                        px, py = map(int,pose_xy[pi,k])
                        cv2.circle(frame,(px,py),3,(255,255,0),-1)
            if inside and looking is True:
                look_frames[track_id] += 1
                if look_frames[track_id]/fps >= MIN_LOOK_SECONDS:
                    look_ids.add(track_id)

            if inside and dwell_seconds >= MIN_DWELL_SECONDS:
                status, color = "DWELL", (0,255,0)
            elif inside: status, color = "FOOTFALL", (0,255,255)
            else: status, color = "OUTSIDE", (255,0,0)
            look_label = "FACING SCREEN" if looking is True else ("AWAY" if looking is False else "UNKNOWN")
            cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
            cv2.circle(frame,foot_point,4,color,-1)
            cv2.putText(frame,f"ID:{track_id} {status} {dwell_seconds:.1f}s",
                        (x1,max(y1-25,18)),cv2.FONT_HERSHEY_SIMPLEX,0.55,color,2)
            cv2.putText(frame,f"Head: {look_label}",
                        (x1,max(y1-5,38)),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),2)
            if head is not None and looking is True:
                cv2.line(frame,tuple(head.astype(int)),tuple(SCREEN_CENTER.astype(int)),(0,200,0),1)

    dwell_ids = {tid for tid,nf in frames_inside.items() if nf/fps >= MIN_DWELL_SECONDS}
    for j, text in enumerate([
        f"Footfall: {len(footfall_ids)}",
        f"Dwell events: {len(dwell_ids)}",
        f"Facing-screen events: {len(look_ids)}",
        f"Dwell threshold: {MIN_DWELL_SECONDS:.1f}s"]):
        cv2.putText(frame,text,(30,45+36*j),cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,(255,255,255),2)
    writer.write(frame)

cap.release()
writer.release()


In [ ]:
dwell_ids = {tid for tid, frames in frames_inside.items()
             if frames/fps >= MIN_DWELL_SECONDS}
print("Footfall:", len(footfall_ids))
print("Dwell events:", len(dwell_ids))
print("Estimated screen-facing events:", len(look_ids))
print("Screen-facing heuristic requires visible nose and eye/ear pair;")
print("UNKNOWN does not count as a screen-facing event.")
print("\nPer-person metrics:")
for tid in sorted(footfall_ids):
    print(f"ID {tid}: dwell={frames_inside[tid]/fps:.2f}s | "
          f"screen-facing={look_frames[tid]/fps:.2f}s | "
          f"facing-event={tid in look_ids}")
print("Saved video:", OUTPUT_VIDEO)


In [ ]:
from google.colab import files
files.download(OUTPUT_VIDEO)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
rows = []
for track_id in sorted(footfall_ids):
    dwell_time = frames_inside[track_id] / fps
    facing_time = look_frames[track_id] / fps
    rows.append({
        "person_id": track_id,
        "footfall": True,
        "dwell_time_seconds": round(dwell_time,2),
        "dwell_event": dwell_time >= MIN_DWELL_SECONDS,
        "estimated_screen_facing_seconds": round(facing_time,2),
        "estimated_screen_facing_event": track_id in look_ids
    })
df = pd.DataFrame(rows)
df
